In [ ]:
import torch
import functools
import einops
import requests
import pandas as pd
import io
import textwrap
import gc
import json

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch import Tensor
from typing import List, Callable
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
from transformers import AutoTokenizer
from jaxtyping import Float, Int
from typing import List, Tuple, Callable
import transformer_lens
import contextlib

In [ ]:
def load_judgment_data(model_name: str, judge_type: str) -> dict:

    clean_name = model_name.replace("/", "_").replace(":", "_").replace("@", "_")
    
    if judge_type == "safety":
        file_path = f"artifacts/responses/safety_judge/{clean_name}/safety_judge_results.json"
    elif judge_type == "self":
        file_path = f"artifacts/responses/self_judge/{clean_name}/gaslighting_self_judge_results.json"
    else:
        raise ValueError("judge_type must be either 'safety' or 'self'")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)


In [ ]:
model_name = "meta-llama/Llama-2-7b-chat-hf"
# model_name = "Qwen/Qwen3-1.7B"
# model_name = "Qwen/Qwen3-4B"

In [ ]:
type_ = ["self", "safety"]

harmful_ids = {}
for type_name in type_:
    data = load_judgment_data(model_name, type_name)
    harmful = []
    for el in data['judgments']:
        if el['judgment'] == 'HARMFUL':
            harmful.append(el['qid'])
    harmful_ids[type_name] = harmful

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a 100x2 matrix
matrix = np.zeros((100, 2))

# Fill the matrix: column 0 for self judge, column 1 for safety judge
for i in range(100):
    if i in harmful_ids['self']:
        matrix[i, 0] = 1
    if i in harmful_ids['safety']:
        matrix[i, 1] = 1

# Plot with imshow
fig, ax = plt.subplots()
im = ax.imshow(matrix, aspect='auto', cmap='Blues', interpolation='nearest')

# Set labels
ax.set_xlabel('Judge Type')
ax.set_ylabel('Question ID')
ax.set_title(f'{model_name}')
ax.set_xticks([0, 1])
ax.set_xticklabels([f'Self Judge\n({len(harmful_ids["self"])}/100)', f'Safety Judge\n({len(harmful_ids["safety"])}/100)'])
ax.set_yticks(range(0, 100, 10))

# Add colorbar
plt.colorbar(im, ax=ax, label='Harmful (1) / Safe (0)')

# Add text with overlap information
overlap = list(set(harmful_ids['self']) & set(harmful_ids['safety']))

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"Self judge harmful ({len(harmful_ids['self'])}/100): {harmful_ids['self']}")
print(f"Safety judge harmful ({len(harmful_ids['safety'])}/100): {harmful_ids['safety']}")
overlap = list(set(harmful_ids['self']) & set(harmful_ids['safety']))
print(f"Both judges agree harmful ({len(overlap)}/100): {overlap}")

In [ ]:
import torch

model_name = 'meta-llama/Llama-2-7b-chat-hf'
# model_name = 'meta-llama/Llama-3.1-8B-Instruct'
# model_name = 'Qwen/Qwen3-4B'
# model_name = 'Qwen/Qwen3-1.7B'
clean = model_name.replace("/", "_").replace(":", "_").replace("@", "_")
chat = f'artifacts/residuals/{clean}/chat_completion_residuals.pt'
gen = f'artifacts/residuals/{clean}/completion_residuals.pt'
resid_chat = torch.load(chat)
resid_gen = torch.load(gen)

In [ ]:
len(resid_chat['pre'])

In [ ]:
chat_pre = []
gen_pre = []
for i in range(len(resid_chat['pre'])):
    chat_pre_i = resid_chat['pre'][i][:, -1]
    chat_pre.append(chat_pre_i)

    gen_pre_i = resid_gen['pre'][i][:, -1]
    gen_pre.append(gen_pre_i)

chat_pre = torch.stack(chat_pre)
gen_pre = torch.stack(gen_pre)

In [ ]:
chat_pre.shape, gen_pre.shape

In [ ]:
bad = resid_gen['pre'][1][:, 42]
good = resid_gen['pre'][1][:, 41]

In [ ]:
bad.shape, good.shape, chat_pre.shape, gen_pre.shape

In [ ]:
# Append bad to gen_pre and good to chat_pre
gen_pre = torch.cat([gen_pre, bad.unsqueeze(0)], dim=0)
chat_pre = torch.cat([chat_pre, good.unsqueeze(0)], dim=0)

In [ ]:
gen_pre.shape, chat_pre.shape

In [ ]:
all_trajectories_tensor.shape, reshaped_for_pca.shape, projected_trajectories.shape

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

print(f"Shape of chat_pre: {chat_pre.shape}")
print(f"Shape of gen_pre: {gen_pre.shape}")

# Combine both sets of trajectories into a single tensor for PCA
# Shape will be (200, 32, 4096)
all_trajectories_tensor = torch.cat([chat_pre, gen_pre], dim=0)

num_trajectories, num_layers, hidden_dim = all_trajectories_tensor.shape
reshaped_for_pca = all_trajectories_tensor.reshape(-1, hidden_dim)

reshaped_for_pca_np = reshaped_for_pca

print("\nFitting PCA on all 6400 trajectory points...")
pca = PCA(n_components=2)
projected_trajectories = pca.fit_transform(reshaped_for_pca_np)

projected_trajectories = projected_trajectories.reshape(num_trajectories, num_layers, -1)
print("PCA fitting complete.")

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# --- 3. Visualization ---
print("Generating plot...")

n = 101

# Create traces for the plot
traces = []

# Plot the 100 "Refusal" trajectories
for i in range(n):
    trajectory = projected_trajectories[i]
    if i == n-1:

        # Add trajectory line
        traces.append(go.Scatter(
            x=trajectory[:, 0],
            y=trajectory[:, 1],
            mode='lines',
            line=dict(color='green', width=10),
            opacity=0.2,
            name='chat' if i == 0 else '',
            showlegend=True if i == 0 else False,
            legendgroup='chat',
            hovertemplate=f'Chat Trajectory {i}<br>PC1: %{{x}}<br>PC2: %{{y}}<extra></extra>'
        ))
    else:
        # Add trajectory line
        traces.append(go.Scatter(
            x=trajectory[:, 0],
            y=trajectory[:, 1],
            mode='lines',
            line=dict(color='crimson', width=1),
            opacity=0.2,
            name='chat' if i == 0 else '',
            showlegend=True if i == 0 else False,
            legendgroup='chat',
            hovertemplate=f'Chat Trajectory {i}<br>PC1: %{{x}}<br>PC2: %{{y}}<extra></extra>'
        ))
    
    # Add start point (layer 0)
    traces.append(go.Scatter(
        x=[trajectory[0, 0]],
        y=[trajectory[0, 1]],
        mode='markers',
        marker=dict(color='crimson', size=4),
        opacity=0.3,
        name='',
        showlegend=False,
        hovertemplate=f'Chat {i} - Start (Layer 0)<br>PC1: %{{x}}<br>PC2: %{{y}}<extra></extra>'
    ))
    
    # Add end point (layer 31)
    traces.append(go.Scatter(
        x=[trajectory[-1, 0]],
        y=[trajectory[-1, 1]],
        mode='markers',
        marker=dict(color='black', symbol='x', size=8),
        opacity=0.4,
        name='',
        showlegend=False,
        hovertemplate=f'Chat {i} - End (Layer 31)<br>PC1: %{{x}}<br>PC2: %{{y}}<extra></extra>'
    ))

# Plot the 100 "Harmful Continuation" trajectories
for i in range(101, 101+n):
    if i == 151:
        continue
    
    trajectory = projected_trajectories[i]
    gen_idx = i - 101  # Get the gen index for hover
    if gen_idx == n-1:
        # Add trajectory line
        traces.append(go.Scatter(
            x=trajectory[:, 0],
            y=trajectory[:, 1],
            mode='lines',
            line=dict(color='yellow', width=10),
            opacity=0.2,
            name='gen' if i == 101 else '',
            showlegend=True if i == 101 else False,
            legendgroup='gen',
            hovertemplate=f'Gen Trajectory {gen_idx}<br>PC1: %{{x}}<br>PC2: %{{y}}<extra></extra>'
        ))
    else:
        # Add trajectory line
        traces.append(go.Scatter(
            x=trajectory[:, 0],
            y=trajectory[:, 1],
            mode='lines',
            line=dict(color='royalblue', width=1),
            opacity=0.2,
            name='gen' if i == 101 else '',
            showlegend=True if i == 101 else False,
            legendgroup='gen',
            hovertemplate=f'Gen Trajectory {gen_idx}<br>PC1: %{{x}}<br>PC2: %{{y}}<extra></extra>'
        ))
    
    # Add start point (layer 0)
    traces.append(go.Scatter(
        x=[trajectory[0, 0]],
        y=[trajectory[0, 1]],
        mode='markers',
        marker=dict(color='royalblue', size=4),
        opacity=0.3,
        name='',
        showlegend=False,
        hovertemplate=f'Gen {gen_idx} - Start (Layer 0)<br>PC1: %{{x}}<br>PC2: %{{y}}<extra></extra>'
    ))
    
    # Add end point (layer 31)
    traces.append(go.Scatter(
        x=[trajectory[-1, 0]],
        y=[trajectory[-1, 1]],
        mode='markers',
        marker=dict(color='black', symbol='x', size=8),
        opacity=0.4,
        name='',
        showlegend=False,
        hovertemplate=f'Gen {gen_idx} - End (Layer 31)<br>PC1: %{{x}}<br>PC2: %{{y}}<extra></extra>'
    ))

# Create the figure
fig = go.Figure(data=traces)

# Update layout
fig.update_layout(
    title='PCA of Hidden State Trajectories (Layer 0 to 31)',
    xaxis_title='Principal Component 1',
    yaxis_title='Principal Component 2',
    width=800,
    height=500,
    showlegend=True,
    hovermode='closest'
)

# Add grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

fig.show()

In [ ]:
# pip install umap-learn scikit-learn matplotlib
import os, numpy as np, torch, umap
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

out_dir = "./umap_layer_plots"
os.makedirs(out_dir, exist_ok=True)

# --- inputs ---
# chat_pre: torch.Size([100, 32, 4096])
# gen_pre : torch.Size([100, 32, 4096])
assert chat_pre.shape == gen_pre.shape
N, L, D = chat_pre.shape
assert N == 100 and L == 32  # as you described

def to_np(x: torch.Tensor) -> np.ndarray:
    return x.detach().to("cpu", dtype=torch.float32).numpy()

silhouette_by_layer = []

for i in range(L):
    i = 31
    # Collect 100 from each class at layer i -> 200 x 4096
    X_chat = to_np(chat_pre[:, i, :])  # (100, 4096)
    X_gen  = to_np(gen_pre [:, i, :])  # (100, 4096)

    X = np.vstack([X_chat, X_gen])     # (200, 4096)
    y = np.array([0]*N + [1]*N)        # 0=chat, 1=gen

    # UMAP (cosine tends to work well for rep spaces)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
    Z = reducer.fit_transform(X)       # (200, 2)

    # Silhouette on 2D embedding (higher = better separation)
    sil = silhouette_score(Z, y, metric="euclidean")
    silhouette_by_layer.append(sil)

    # Plot
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.scatter(Z[y==0, 0], Z[y==0, 1], s=18, alpha=0.85, label="chat")
    ax.scatter(Z[y==1, 0], Z[y==1, 1], s=18, alpha=0.85, label="gen", marker="^")
    ax.set_title(f"UMAP – Layer {i}  |  silhouette={sil:.3f}")
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    ax.legend(loc="best", frameon=True)
    ax.grid(True, linewidth=0.4, alpha=0.4)

    fig.tight_layout()
    # fig.savefig(os.path.join(out_dir, f"umap_layer_{i:02d}.png"), dpi=160)
    break    

In [ ]:
# pip install umap-learn scikit-learn plotly
import os, numpy as np, torch, umap
import plotly.graph_objects as go
from sklearn.metrics import silhouette_score

out_dir = "./umap_layer_plots"
os.makedirs(out_dir, exist_ok=True)

# --- inputs ---
# chat_pre: torch.Size([100, 32, 4096])
# gen_pre : torch.Size([100, 32, 4096])
assert chat_pre.shape == gen_pre.shape
N, L, D = chat_pre.shape
assert N == 100 and L == 32  # as you described

def to_np(x: torch.Tensor) -> np.ndarray:
    return x.detach().to("cpu", dtype=torch.float32).numpy()

# Choose specific layer l
l = 30  # Change this to the layer you want to visualize

# Collect 100 from each class at layer l -> 200 x 4096
X_chat = to_np(chat_pre[:, l, :])  # (100, 4096)
X_gen  = to_np(gen_pre [:, l, :])  # (100, 4096)

X = np.vstack([X_chat, X_gen])     # (200, 4096)
y = np.array([0]*N + [1]*N)        # 0=chat, 1=gen

# UMAP (cosine tends to work well for rep spaces)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
Z = reducer.fit_transform(X)       # (200, 2)

# Silhouette on 2D embedding (higher = better separation)
sil = silhouette_score(Z, y, metric="euclidean")

# Create index arrays for hover text
chat_indices = list(range(100))  # 0 to 99
gen_indices = list(range(100))   # 0 to 99

# Create plotly figure
fig = go.Figure()

# Add chat data points
fig.add_trace(go.Scatter(
    x=Z[y==0, 0], 
    y=Z[y==0, 1],
    mode='markers',
    marker=dict(size=8, opacity=0.85),
    name='chat',
    text=[f"chat_{i}" for i in chat_indices],
    hovertemplate='<b>%{text}</b><br>UMAP-1: %{x}<br>UMAP-2: %{y}<extra></extra>'
))

# Add gen data points
fig.add_trace(go.Scatter(
    x=Z[y==1, 0], 
    y=Z[y==1, 1],
    mode='markers',
    marker=dict(size=8, opacity=0.85, symbol='triangle-up'),
    name='gen',
    text=[f"gen_{i}" for i in gen_indices],
    hovertemplate='<b>%{text}</b><br>UMAP-1: %{x}<br>UMAP-2: %{y}<extra></extra>'
))

# Update layout
fig.update_layout(
    title=f"UMAP – Layer {l}  |  silhouette={sil:.3f}",
    xaxis_title="UMAP-1",
    yaxis_title="UMAP-2",
    width=700,
    height=550,
    showlegend=True
)

# Save and show
# fig.write_html(os.path.join(out_dir, f"umap_layer_{l:02d}.html"))
fig.show()

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
i = 1
resid1 = resid_chat['pre'][i]
resid2 = resid_gen['pre'][i]

cosine_sim = F.cosine_similarity(resid1, resid2, dim=-1)
plt.imshow(cosine_sim)
plt.colorbar()
plt.show()

In [ ]:
import json
import matplotlib.pyplot as plt

# model_name = 'meta-llama/Llama-2-7b-chat-hf'
# model_name = 'Qwen/Qwen3-4B-Instruct-2507'
model_name = 'meta-llama/Llama-3.1-8B-Instruct'

clean_model_name = model_name.replace("/", "_").replace(":", "_").replace("@", "_")

file = f'artifacts/responses/gaslighting_generation/{clean_model_name}/gpt4o_labels.json'

with open(file, 'r') as f:
    data = json.load(f)

labels = [item['label_4o'] for item in data]
disclaimers = [item['disclaimer'] for item in data]

In [ ]:
for model_name in ['meta-llama/Llama-2-7b-chat-hf', 'Qwen/Qwen3-4B-Instruct-2507', 'meta-llama/Llama-3.1-8B-Instruct']:
    clean_model_name = model_name.replace("/", "_").replace(":", "_").replace("@", "_")

    file = f'artifacts/responses/gaslighting_generation/{clean_model_name}/gpt4o_labels.json'

    with open(file, 'r') as f:
        data = json.load(f)

    labels = [item['label_4o'] for item in data]
    disclaimers = [item['disclaimer'] for item in data]

    # Convert labels to binary: harmful = 1, other = 0
    labels_binary = [1 if label == 'harmful' else 0 for label in labels]
    
    # Convert disclaimers to binary: yes = 1, no = 0
    disclaimers_binary = [1 if disclaimer == 'yes' else 0 for disclaimer in disclaimers]

    print(f'Model: {model_name}')
    print(f' how many harmful? {sum(labels_binary)} / {len(labels_binary)}')
    print(f' how many disclaimers? {sum(disclaimers_binary)} / {len(disclaimers_binary)}')
    # print(f' total? {len(labels)}')
    print('')

In [ ]:
from datasets import load_dataset

ds = load_dataset("MuskumPillerum/General-Knowledge")

In [ ]:
ds['train'][0]

In [ ]:
# Create a JSON dataset from the ds dataset with question-response pairs
import json

# Create the output list
output_data = []
n = 200
# Process each item in the dataset
for i in range(min(n, len(ds['train']))):
    item = ds['train'][i]
    # Extract question and answer
    question = item['Question']
    response = item['Answer']
    
    # Create the dictionary entry
    entry = {
        "question": question,
        "response": response
    }
    
    # Add to output list
    output_data.append(entry)

# # Save to JSON file
with open('artifacts/data/qa_harmless.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

# print(f"Created JSON dataset with {len(output_data)} question-response pairs")
# print("Sample entry:")
# print(json.dumps(output_data[0], indent=2))

In [ ]:
len(output_data)